In [2]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
data=load_breast_cancer()

In [3]:
x=data.data
y=data.target

In [4]:
print(x.shape)
print(y.shape)
print(np.unique(y))
print(data.feature_names)

(569, 30)
(569,)
[0 1]
['mean radius' 'mean texture' 'mean perimeter' 'mean area'
 'mean smoothness' 'mean compactness' 'mean concavity'
 'mean concave points' 'mean symmetry' 'mean fractal dimension'
 'radius error' 'texture error' 'perimeter error' 'area error'
 'smoothness error' 'compactness error' 'concavity error'
 'concave points error' 'symmetry error' 'fractal dimension error'
 'worst radius' 'worst texture' 'worst perimeter' 'worst area'
 'worst smoothness' 'worst compactness' 'worst concavity'
 'worst concave points' 'worst symmetry' 'worst fractal dimension']


In [5]:
df = pd.DataFrame(
    data=x,
    columns=[
        'mean radius','mean texture','mean perimeter','mean area',
        'mean smoothness','mean compactness','mean concavity',
        'mean concave points','mean symmetry','mean fractal dimension',
        'radius error','texture error','perimeter error','area error',
        'smoothness error','compactness error','concavity error',
        'concave points error','symmetry error','fractal dimension error',
        'worst radius','worst texture','worst perimeter','worst area',
        'worst smoothness','worst compactness','worst concavity',
        'worst concave points','worst symmetry','worst fractal dimension'
    ]
)

In [6]:
df['target']=y
df

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.30010,0.14710,0.2419,0.07871,...,17.33,184.60,2019.0,0.16220,0.66560,0.7119,0.2654,0.4601,0.11890,0
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.08690,0.07017,0.1812,0.05667,...,23.41,158.80,1956.0,0.12380,0.18660,0.2416,0.1860,0.2750,0.08902,0
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.19740,0.12790,0.2069,0.05999,...,25.53,152.50,1709.0,0.14440,0.42450,0.4504,0.2430,0.3613,0.08758,0
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.24140,0.10520,0.2597,0.09744,...,26.50,98.87,567.7,0.20980,0.86630,0.6869,0.2575,0.6638,0.17300,0
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.19800,0.10430,0.1809,0.05883,...,16.67,152.20,1575.0,0.13740,0.20500,0.4000,0.1625,0.2364,0.07678,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
564,21.56,22.39,142.00,1479.0,0.11100,0.11590,0.24390,0.13890,0.1726,0.05623,...,26.40,166.10,2027.0,0.14100,0.21130,0.4107,0.2216,0.2060,0.07115,0
565,20.13,28.25,131.20,1261.0,0.09780,0.10340,0.14400,0.09791,0.1752,0.05533,...,38.25,155.00,1731.0,0.11660,0.19220,0.3215,0.1628,0.2572,0.06637,0
566,16.60,28.08,108.30,858.1,0.08455,0.10230,0.09251,0.05302,0.1590,0.05648,...,34.12,126.70,1124.0,0.11390,0.30940,0.3403,0.1418,0.2218,0.07820,0
567,20.60,29.33,140.10,1265.0,0.11780,0.27700,0.35140,0.15200,0.2397,0.07016,...,39.42,184.60,1821.0,0.16500,0.86810,0.9387,0.2650,0.4087,0.12400,0


In [7]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X = scaler.fit_transform(x)
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [9]:
central_model=LogisticRegression(max_iter=1000)
central_model.fit(X_train,y_train)

y_pred=central_model.predict(X_test)
central_accuracy=accuracy_score(y_test,y_pred)
print("Centralized Accuracy:",central_accuracy)

Centralized Accuracy: 0.9736842105263158


In [10]:
hospital_A_x = X[0:190]
hospital_A_y=y[0:190]
hospital_B_x = X[190:380]
hospital_B_y=y[190:380]
hospital_C_x = X[380:]
hospital_C_y=y[380:]

In [11]:
print(f"Hospital A: {hospital_A_x.shape[0]} patients")
print(f"Hospital B: {hospital_B_x.shape[0]} patients")
print(f"Hospital C: {hospital_C_x.shape[0]} patients")
print(f"Total: {hospital_A_x.shape[0] + hospital_B_x.shape[0] + hospital_C_x.shape[0]} patients")

Hospital A: 190 patients
Hospital B: 190 patients
Hospital C: 189 patients
Total: 569 patients


In [12]:
def train_local_model(x_local, y_local, global_model):
    local_model = LogisticRegression(max_iter=1000)

    # load global weights
    local_model.coef_ = global_model.coef_.copy()
    local_model.intercept_ = global_model.intercept_.copy()
    local_model.classes_ = global_model.classes_

    # local training
    local_model.fit(x_local, y_local)

    return local_model.coef_, local_model.intercept_

In [13]:
def federated_average(weights, intercepts):
    avg_w = np.mean(weights, axis=0)
    avg_b = np.mean(intercepts, axis=0)
    return avg_w, avg_b

In [14]:
# Simple security check (AI-side)
# -------------------------------
def is_update_valid(weights, threshold=50):
    # Reject abnormally large updates
    return np.linalg.norm(weights) < threshold

In [15]:
global_model = LogisticRegression(max_iter=1000)

# LogisticRegression needs both classes once
idx_0 = np.where(hospital_A_y == 0)[0][0]
idx_1 = np.where(hospital_A_y == 1)[0][0]

init_x = hospital_A_x[[idx_0, idx_1]]
init_y = hospital_A_y[[idx_0, idx_1]]
global_model.fit(init_x, init_y)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [16]:
print("\nFederated Learning Rounds:")

for round in range(3):
    weights = []
    intercepts = []

    for hx, hy in [
        (hospital_A_x, hospital_A_y),
        (hospital_B_x, hospital_B_y),
        (hospital_C_x, hospital_C_y)]:
          w, b = train_local_model(hx, hy, global_model)

          # Security validation
          if is_update_valid(w):
              weights.append(w)
              intercepts.append(b)
          else:
              print("Suspicious update rejected")
      # Aggregate updates
    global_coef, global_intercept = federated_average(weights, intercepts)
     # Update global model
    global_model.coef_ = global_coef
    global_model.intercept_ = global_intercept

    # Evaluate
    y_pred_fed = global_model.predict(X_test)
    fed_accuracy = accuracy_score(y_test, y_pred_fed)

    print(f"Round {round + 1} Federated Accuracy: {fed_accuracy}")



Federated Learning Rounds:
Round 1 Federated Accuracy: 0.9736842105263158
Round 2 Federated Accuracy: 0.9736842105263158
Round 3 Federated Accuracy: 0.9736842105263158


In [17]:
# -------------------------------
# 10. Final comparison
# -------------------------------
print("\nFinal Comparison:")
print("Centralized Accuracy:", central_accuracy)
print("Federated Accuracy:", fed_accuracy)
print("Difference (Central - Federated):", central_accuracy - fed_accuracy)


Final Comparison:
Centralized Accuracy: 0.9736842105263158
Federated Accuracy: 0.9736842105263158
Difference (Central - Federated): 0.0


In [22]:
# ----------------------------------------
# SAVE TRAINED MODEL WEIGHTS (FOR SECURITY DEMO)
# ----------------------------------------
import numpy as np

model_state = {
    "coef": central_model.coef_,
    "intercept": central_model.intercept_
}

np.save("trained_weights.npy", model_state, allow_pickle=True)

print("✅ Model weights saved successfully for run_demo.py")





✅ Model weights saved successfully for run_demo.py
